<a href="https://colab.research.google.com/github/AditiSinghal28/Machine-Learning-Projects/blob/ml/Deploying_ML_model_as_Public_API_ngrok.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

INSTALLING THE DEPENDENCIES

In [ ]:
!pip install fastapi
!pip install uvicorn
!pip install pickle5
!pip install pydantic
!pip install scikit-learn
!pip install requests
!pip install pypi-json
!pip install pyngrok
!pip install nest-asyncio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.1/132.1 kB 11.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for pickle5
  Running setup.py clean for pickle5
Failed to build pickle5
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (pickle5)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.3/99.3 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.9/126.9 kB 12.4 MB/s eta 0:00:00


In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
import pickle
import json
import uvicorn
from pyngrok import ngrok
from fastapi.middleware.cors import CORSMiddleware
import nest_asyncio

In [ ]:
app = FastAPI()

In [ ]:
origins =["*"]

app.add_middleware(
    CORSMiddleware,
    allow_origins=origins,
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

In [ ]:
class model_input(BaseModel):

    #columns of dataset
     Pregnancies : int
     Glucose : int
     BloodPressure : int
     SkinThickness : int
     Insulin : int
     BMI : float
     DiabetesPedigreeFunction : float
     Age : int

In [ ]:
#loading the saved model

diabetes_model = pickle.load(open('trained_model.sav', 'rb'))

In [ ]:
@app.post('/diabetes_prediction')
def diabetes_pred(input_parameters : model_input) :

    input_data = input_parameters.json()
    input_dictionary = json.loads(input_data)

    preg = input_dictionary['Pregnancies']
    glu = input_dictionary['Glucose']
    bp = input_dictionary['BloodPressure']
    skin = input_dictionary['SkinThickness']
    ins = input_dictionary['Insulin']
    bmi = input_dictionary['BMI']
    dpf = input_dictionary['DiabetesPedigreeFunction']
    age = input_dictionary['Age']


    input_list = [preg, glu, bp, skin, ins, bmi, dpf, age]

    prediction = diabetes_model.predict([input_list])

    if (prediction[0] == 0):
        return 'The person is NOT DIABETIC'
    else :
        return 'THe person is Diabetic'

In [ ]:
# to make url public
nest_asyncio.apply()
ngrok.set_auth_token("39QMnsI6tttr4iPcvszF3CwMcml_75ad9qXgK7DtPARDA68oK")
public_url = ngrok.connect(8000)
print("Public URL:", public_url)

config = uvicorn.Config(app, port=8000)
server = uvicorn.Server(config)

await server.serve()


Public URL: NgrokTunnel: "https://diaphragmatic-janey-impregnably.ngrok-free.dev" -> "http://localhost:8000"


INFO:     Started server process [262]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     2405:201:683a:403a:8499:c8b4:8427:3552:0 - "POST /diabetes_prediction HTTP/1.1" 200 OK


/usr/lib/python3.12/importlib/__init__.py:90: RuntimeWarning: coroutine 'Server.serve' was never awaited
  return _bootstrap._gcd_import(name[level:], package, level)
/tmp/ipython-input-3144204442.py:4: PydanticDeprecatedSince20: The `json` method is deprecated; use `model_dump_json` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  input_data = input_parameters.json()
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [262]
